# CHisIEC NER — Full Pipeline
Chạy trên Kaggle GPU. Code được clone từ GitHub.


In [ ]:
# ── CELL 1: Clone repo & install ─────────────────────────────
!git clone https://github.com/YOUR_USERNAME/chisiec-ner.git
%cd chisiec-ner
!pip install -q -r requirements.txt
print('✅ Ready')

In [ ]:
# ── CELL 2: Check GPU & symlink data ─────────────────────────
import torch, os, shutil
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Symlink dataset vào ./data
DATA_SRC = '/kaggle/input/chisiec'   # <-- đổi tên đúng dataset của bạn
os.makedirs('data', exist_ok=True)
for f in ['train.txt','dev.txt','test.txt']:
    src = os.path.join(DATA_SRC, f)
    dst = os.path.join('data', f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
print('Data files:', os.listdir('data'))

In [ ]:
# ── CELL 3: EDA + Data Validation ────────────────────────────
!python scripts/run_eda.py --data_dir data --output_dir outputs/eda

from IPython.display import Image, display
display(Image('outputs/eda/sentence_lengths.png'))
display(Image('outputs/eda/train_label_dist.png'))
display(Image('outputs/eda/entity_heatmap.png'))

In [ ]:
# ── CELL 4: Train Method 1 — GuwenBERT + CRF ─────────────────
!python scripts/train.py model.method=guwenbert_crf training.fp16=true

In [ ]:
# ── CELL 5: Train Method 2 — RoBERTa + BiLSTM + CRF ──────────
!python scripts/train.py model.method=roberta_bilstm_crf training.fp16=true

In [ ]:
# ── CELL 6: Train Method 3 — RoBERTa + KAN + CRF ─────────────
!python scripts/train.py model.method=roberta_kan_crf training.fp16=true

In [ ]:
# ── CELL 7: Compare all 3 methods ────────────────────────────
!python scripts/evaluate_compare.py --output_dir outputs

from IPython.display import Image, display
display(Image('outputs/compare_test_f1.png'))
display(Image('outputs/compare_training_curves.png'))
display(Image('outputs/compare_entity_f1.png'))

In [ ]:
# ── CELL 8: Inference — predict câu mới ──────────────────────
!python scripts/inference.py \
    --method guwenbert_crf \
    --text '唐太宗李世民於貞觀元年詔令修撰五代史志'

In [ ]:
# ── CELL 9: Xem kết quả tổng hợp ─────────────────────────────
import json, pandas as pd

rows = []
for method in ['guwenbert_crf','roberta_bilstm_crf','roberta_kan_crf']:
    path = f'outputs/{method}/results.json'
    if not __import__('os').path.exists(path): continue
    r = json.load(open(path))
    rows.append({'Method': method, 'Dev F1': r['best_dev_f1'],
                 'Test F1': r['test_f1'], 'Best Epoch': r['best_epoch']})

df = pd.DataFrame(rows)
display(df.style.highlight_max(subset=['Dev F1','Test F1'], color='lightgreen'))